In [2]:
import cv2
import numpy as np

# Try TensorFlow first. If not available, use tflite_runtime.
try:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
except ImportError:
    from tflite_runtime.interpreter import Interpreter


MODEL_PATH = "movenet_lightning.tflite"
INPUT_SIZE = 192
KEYPOINT_THRESHOLD = 0.3


KEYPOINT_DICT = {
    "nose": 0,
    "left_eye": 1,
    "right_eye": 2,
    "left_ear": 3,
    "right_ear": 4,
    "left_shoulder": 5,
    "right_shoulder": 6,
    "left_elbow": 7,
    "right_elbow": 8,
    "left_wrist": 9,
    "right_wrist": 10,
    "left_hip": 11,
    "right_hip": 12,
    "left_knee": 13,
    "right_knee": 14,
    "left_ankle": 15,
    "right_ankle": 16,
}


EDGES = [
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear"),
    ("right_eye", "right_ear"),
    ("nose", "left_shoulder"),
    ("nose", "right_shoulder"),
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle"),
]


def load_model(model_path):
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter


def run_movenet(interpreter, frame_bgr):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    input_image = cv2.resize(frame_rgb, (INPUT_SIZE, INPUT_SIZE))
    input_image = np.expand_dims(input_image, axis=0)

    # MoveNet Lightning INT8 usually expects int32 input.
    input_dtype = input_details[0]["dtype"]

    if input_dtype == np.float32:
        input_image = input_image.astype(np.float32)
    else:
        input_image = input_image.astype(np.int32)

    interpreter.set_tensor(input_details[0]["index"], input_image)
    interpreter.invoke()

    keypoints = interpreter.get_tensor(output_details[0]["index"])

    # Shape: [1, 1, 17, 3]
    # Each keypoint: [y, x, confidence]
    return keypoints[0, 0, :, :]


def draw_keypoints(frame, keypoints):
    h, w, _ = frame.shape

    for i, keypoint in enumerate(keypoints):
        y, x, score = keypoint

        if score < KEYPOINT_THRESHOLD:
            continue

        cx = int(x * w)
        cy = int(y * h)

        cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
        cv2.putText(
            frame,
            str(i),
            (cx + 5, cy - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1
        )


def draw_edges(frame, keypoints):
    h, w, _ = frame.shape

    for start_name, end_name in EDGES:
        start_idx = KEYPOINT_DICT[start_name]
        end_idx = KEYPOINT_DICT[end_name]

        y1, x1, score1 = keypoints[start_idx]
        y2, x2, score2 = keypoints[end_idx]

        if score1 < KEYPOINT_THRESHOLD or score2 < KEYPOINT_THRESHOLD:
            continue

        pt1 = (int(x1 * w), int(y1 * h))
        pt2 = (int(x2 * w), int(y2 * h))

        cv2.line(frame, pt1, pt2, (255, 0, 0), 2)


def body_visible_enough(keypoints):
    required = [
        KEYPOINT_DICT["left_shoulder"],
        KEYPOINT_DICT["right_shoulder"],
        KEYPOINT_DICT["left_hip"],
        KEYPOINT_DICT["right_hip"],
    ]

    visible = 0

    for idx in required:
        score = keypoints[idx][2]
        if score >= KEYPOINT_THRESHOLD:
            visible += 1

    return visible >= 3


def main():
    interpreter = load_model(MODEL_PATH)

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    while True:
        success, frame = cap.read()

        if not success:
            print("Error: Could not read frame.")
            break

        frame = cv2.flip(frame, 1)

        keypoints = run_movenet(interpreter, frame)

        draw_edges(frame, keypoints)
        draw_keypoints(frame, keypoints)

        if body_visible_enough(keypoints):
            label = "Body visible: OK for fall detection"
            color = (0, 255, 0)
        else:
            label = "UNCERTAIN: Body partially visible"
            color = (0, 255, 255)

        cv2.putText(
            frame,
            label,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

        cv2.imshow("MoveNet Lightning Webcam", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'cv2'